# 04 · Algorithms: from a point grid to a brick wall

**Why this is needed:** loops and a conditional rule let a few dimensions describe many repeated parts; a geometric algorithm makes variations reproducible.

**CAS Computational Form-finding · Block Research Group · ETH Zürich**

An algorithm turns a few parameters into many related objects. First build a
point grid with loops. Then add a conditional row offset and create a staggered
wall from native COMPAS boxes and meshes. This is a geometric pattern, without
an assessment of masonry equilibrium. Dimensions are in metres.

**Predict:** how many points does a grid with 5 columns and 4 rows contain?


## 1. Start Python
Run all cells in order with **Run → Run All Cells**, or step through with **Shift+Enter**.
JupyterLite supplies NumPy and SciPy. On desktop Jupyter, install them with
`%pip install -q numpy scipy` if an import below reports a missing package.
The pinned COMPAS packages below are the course's tested versions; `--no-deps`
avoids installing a desktop notebook server in browser Python.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps


In [ ]:
# Enable widgets on Colab; browser JupyterLite and desktop Jupyter need no change.
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


In [ ]:
import numpy as np
from compas.geometry import Box, Frame, Pointcloud
from compas_notebook import Viewer
from compas_notebook.config import Config


## 2. Make a grid with nested loops
`range(n)` supplies indices `0` to `n - 1`. The outer loop changes rows; the inner
loop changes columns. Store the point data first, then ask the viewer to draw it.


In [ ]:
n_columns = 5
n_rows = 4
spacing_x = 0.5  # m
spacing_z = 0.25  # m
points = []
for row in range(n_rows):
    for column in range(n_columns):
        points.append([column * spacing_x, 0, row * spacing_z])
point_grid = Pointcloud(points)
assert len(points) == n_columns * n_rows
assert len(set(tuple(xyz) for xyz in points)) == len(points)
print(f"Grid: {len(points)} points")

config = Config()
config.view.width = 700
config.view.height = 360
config.view.show_grid = False

config.view.camera.position = [3, -5, 2]
config.view.camera.target = [1, 0, 0.5]
view = Viewer(config=config)
view.scene.add(point_grid, color="#5552bb", pointsize=0.08)
view.show()


## 3. Add a rule: odd rows shift by half a brick
The remainder operator `%` distinguishes even and odd rows. The frame locates
each box centre; dimensions describe the brick itself. The gap is empty space,
not an extra mesh. Every row contains the same number of whole bricks, so alternate
rows extend beyond the row below; the wall ends are intentionally untrimmed.


In [ ]:
brick_length = 0.40
brick_depth = 0.20
brick_height = 0.15
gap = 0.01
assert min(brick_length, brick_depth, brick_height) > 0 and gap >= 0
pitch = brick_length + gap
bricks = []
brick_meshes = []
for row in range(n_rows):
    offset = (row % 2) * pitch / 2
    for column in range(n_columns):
        centre = [column * pitch + offset, 0,
                  brick_height / 2 + row * (brick_height + gap)]
        frame = Frame(centre, [1, 0, 0], [0, 1, 0])
        brick = Box(brick_length, brick_depth, brick_height, frame=frame)
        bricks.append(brick)
        brick_meshes.append(brick.to_mesh())
print(f"Wall: {len(bricks)} bricks")
print(f"Solid brick volume: {sum(brick.volume for brick in bricks):.4f} m³")


## 4. Check the rule before displaying the objects
An algorithm should satisfy a rule for every row, not just reproduce one picture.
We check counts, volume and the first brick's offset on each row.


In [ ]:
assert len(bricks) == n_rows * n_columns
expected_volume = n_rows * n_columns * brick_length * brick_depth * brick_height
assert np.isclose(sum(brick.volume for brick in bricks), expected_volume)
for row in range(n_rows):
    first = bricks[row * n_columns]
    assert np.isclose(first.frame.point.x, (row % 2) * pitch / 2)
assert all(mesh.number_of_faces() == 6 for mesh in brick_meshes)

wall_view = Viewer(config=config)
for i, mesh in enumerate(brick_meshes):
    colour = "#c78558" if (i // n_columns) % 2 == 0 else "#e4b98e"
    wall_view.scene.add(mesh, color=colour, show_edges=True)
wall_view.show()


## 5. Predict → change → check
Starting result: **20 points, 20 bricks, 0.24 m³ solid brick volume**.

1. Double the number of rows. Predict the new count and volume, then rerun the construction cells.
2. Change `(row % 2) * pitch / 2` to zero. How does the bonding pattern change?
3. Increase the gap. Explain why the bounding size changes while the solid volume does not.
4. Design a three-row repeating offset using `% 3`. Update the offset check to express your new rule.


In [ ]:
# Safe experiment: the offset sequence for a three-row repeat (no scene mutation).
trial_offsets = [(row % 3) * pitch / 3 for row in range(6)]
assert np.allclose(trial_offsets[:3], trial_offsets[3:])
print("Three-row repeat offsets [m]:", trial_offsets)


## Sources
Original course exercise inspired by [Seoul 2025 COMPAS geometry](https://github.com/BRG-teaching/seoul_2025/tree/b641b732a3e337eb3a19d7a91048946820695e7a) and
[Calgary 2025 introduction](https://github.com/BRG-teaching/Calgary2025/tree/05cb4f76aa7ff43a113f0f6ff494970d5c7756c4).
Native objects: [COMPAS geometry](https://compas.dev/compas/2.15.1/api/compas.geometry.html); viewer:
[COMPAS JupyterLite](https://github.com/compas-dev/compas-jupyterlite/tree/d9df5aef10685c80507236554900d72b86504d6a).

Save with **Ctrl+S** (or **⌘S**), then use **File → Save and Export Notebook As → Notebook** to download your work. Browser files are local to this browser; export a copy before moving machines.
